In [2]:
import dask
import time
import toolviper
import random
import webbrowser
import pathlib
import distributed
# ====== Testing ====== #

import asyncio

# ===================== #


import toolviper.utils.logger as logger
import toolviper.utils.display as display

import toolviper.dask.client as client

from collections import defaultdict

from xradio.measurement_set import convert_msv2_to_processing_set
from xradio.measurement_set import open_processing_set

import graphviper.graph_tools as graph_tools

In [3]:
def pipeline_a(x):
    import time
    time.sleep(10)  # Simulating heavy work
    return x * 2

def pipeline_b(x):
    import time
    time.sleep(10)
    return x + 100

async def manage_jobs():
    async with client.local_client(
        cores=12,
        asynchronous=True,
        log_params={
            "log_to_file":False,
            "log_to_term":True,
            "log_level":"DEBUG" 
        },
        worker_log_params={
            "log_to_file":False,
            "log_to_term":True,
            "log_level":"INFO" 
        }
    )as c:
        logger.info(c.dashboard_link) 
        logger.info("Submitting both graphs to the cluster...")
        
        # 2. Submit graph 1 (runs in the background)
        future_a = c.submit(pipeline_a, 10)
        
        # 3. Submit graph 2 (runs in the background at the same time)
        future_b = c.submit(pipeline_b, 10)
        
        # 4. Use asyncio.gather to await both concurrently
        # client.gather is a coroutine when client is asynchronous
        results = await asyncio.gather(
            c.gather(future_a),
            c.gather(future_b)
        )

async def submit():
    await manage_jobs()

In [4]:
result = await submit()

[2026-07-03 12:36:49,699]  WARNING      client:  It is recommended that the local cache directory be set using the dask_local_dir parameter. 
[2026-07-03 12:36:49,725]    DEBUG      client:  Loading plugin module: <class 'worker.DaskWorker'>
[2026-07-03 12:36:49,726]     INFO      client:  Client <MenrvaClient: No scheduler connected> 


/Users/joshua/Development/toolviper-i42/src/toolviper/dask/client.py:269: RuntimeWarning: coroutine 'Client._get_versions' was never awaited
  client.get_versions(check=True)
/Users/joshua/Development/toolviper-i42/src/toolviper/dask/client.py:273: RuntimeWarning: coroutine 'Cluster._wait_for_workers' was never awaited
  client.wait_for_workers(n_workers=cores)
/Users/joshua/Development/toolviper-i42/src/toolviper/dask/menrva.py:198: RuntimeWarning: coroutine 'Client._register_worker_plugin' was never awaited
  client.register_plugin(plugin_instance, name=name)


http://127.0.0.1:8787/status
Submitting both graphs to the cluster...


In [ ]:
client = client.local_client(
    cores=12,
    asynchronous=True,
    log_params={
        "log_to_file":False,
        "log_to_term":True,
        "log_level":"DEBUG" 
    },
    worker_log_params={
        "log_to_file":False,
        "log_to_term":True,
        "log_level":"INFO" 
    }
)

In [ ]:
client.dashboard_link

In [ ]:
toolviper.utils.data.download(file="absolute_off", folder="data")

In [ ]:
def generate_delay(n=1, m=2):
    time.sleep(random.uniform(n, m))

def args_checker(args):
    if "delay" in args[0].keys():
        if args[0]['task_coords']['antenna_name']['data'] == args[0]['delay']:
            return True

    return False

def write_image(file_name, spw, antenna, field):

    path = pathlib.Path(file_name)
    logger.info(str(path))
    if not path.exists():
        logger.error(f"File: {str(path)} not found.")

    data_path = path.joinpath(spw).joinpath(field).joinpath(antenna)
    logger.info(str(data_path))
    data_path.mkdir(exist_ok=True, parents=True)
    
    with open(str(data_path.joinpath(f"some_data_{spw}.{field}.{antenna}.image.ps.zarr")), "w") as file:
        file.write(f"{spw}\t{field}\t{antenna}")

        

### Make Test Functions 

In [ ]:
UPPER_DELAY_LIMIT=2

# Distributed #[field, spw, antenna][EB]
def deviation_mask_heuristic(*args, **kwargs):
    logger.info("Deviation mask heuristic ... [field, spw, antenna]")
    generate_delay(n=1, m=UPPER_DELAY_LIMIT)

# Distributed #[field,spw][EB]
def line_analysis(*args, **kwargs):
    logger.info("Line analysis... [field, spw]")
    generate_delay(n=1, m=UPPER_DELAY_LIMIT)

# Distributed #[field,spw, antenna, pol][EB]
def fitting_heuristic(*args, **kwargs):
    logger.info("Fitting Heuristic ... [field, spw, antenna, pol]")
    generate_delay(n=1, m=UPPER_DELAY_LIMIT)

# Distributed #[field,spw, antenna, pol][EB]
def baseline_subtraction(*args, **kwargs):
    logger.info("Baseline subtraction ... [field, spw, antenna, pol]")
    generate_delay(n=1, m=UPPER_DELAY_LIMIT)

# Distributed #[field,spw, antenna, pol][EB]
def heuristic_plots_per_eb(*args, **kwargs):
    logger.info("Pre-EB heuristic plotting ... [field, spw, antenna, pol]")
    generate_delay(n=1, m=UPPER_DELAY_LIMIT)

# Distributed #[field, spw]
def heuristic_plots(*args, **kwargs):
    logger.info("Heuristic plotting ... [field, spw]")
    generate_delay(n=1, m=UPPER_DELAY_LIMIT)

### Convert the MSv2 to a Processing Set and Open

In [ ]:
file_path = pathlib.Path().cwd().joinpath("../docs/takeshi/uid___A002_Xc1469f_X1c33.ms")
processing_set = pathlib.Path().cwd().joinpath("../docs/takeshi/uid___A002_Xc1469f_X1c33.ms.ps.zarr")

if not processing_set.exists():
    convert_msv2_to_processing_set(
        in_file=str(file_path),
        out_file=str(processing_set)
    )


In [ ]:
ps = open_processing_set(
    ps_store=str(processing_set),
    scan_intents=["OBSERVE_TARGET#ON_SOURCE"]
)

In [ ]:
ps

### Demonstrate the Simple Node Building Tool

In [ ]:
graph = toolviper.utils.sd.graph.Graph.from_dataset(ps)

In [ ]:
graph.filter(leaves=["uid___A002_Xc1469f_X1c33_65", "uid___A002_Xc1469f_X1c33_66"])

In [ ]:
graph.datatree

In [ ]:
graph.make_coordinates(coords=["antenna_name", "field_name"])

In [ ]:
graph.coordinates

In [ ]:
graph.build_node(ps_partition = ["spectral_window_name", "field_name"])

In [ ]:
graph.node_mapping

In [ ]:
graph.map(function=per_imaging_generation, parameters={"parameter": False}, connect=None)             # Adding make_workflow=True here will alllow you to produce results without a gather.
graph.reduce(parameters={"parameter": False}, function=combine_per_antenna_stokes, mode="tree")

In [ ]:
graph.node_mapping

In [ ]:
dask.visualize(graph._graph, filename="map_graph")

In [ ]:
graph.reset()

### Example of `hsd_imaging()` using GraphViper Framework

In [ ]:
# Spawn dashboard window in a seperate tab,
# comment out if you don't want this to spawn.
webbrowser.open(url=client.dashboard_link)

def hsd_imaging(processing_set, leaves=None, file_name="no_name_test.zarr"):
    # Serial processing
    input_imaging_parameters(file_name=file_name)
    set_group()

    # Begin Preprocessing
    ps = open_processing_set(
        ps_store=str(processing_set),
        scan_intents=["OBSERVE_TARGET#ON_SOURCE"]
    )

    job = toolviper.utils.sd.graph.Graph.from_dataset(ps)

    # If you want to filter sub-dataset
    if leaves:
        job.filter(leaves=leaves)

    # We are actually doing [field, spw, antenna] here but the processing set is already split up by spw.
    job.make_coordinates(coords=["antenna_name", "field_name"])
    job.build_node(ps_partition = ["spectral_window_name", "field_name"])
    
    # Start Dask processing - building job graph
    job.map(
        function=per_imaging_generation, 
        parameters={
        "delay": "PM03_T701",
        "file_name": file_name
        },
        connect=None, 
        make_workflow=False
    )
    
    job.reduce(
        parameters={"delay": 'PM03_T701'}, 
        function=combine_per_antenna_stokes, 
        mode="single_node"
    )

    # Finish Dask processing - compute job graph
    job.compute()
    
    make_weblog_entry()
    

In [ ]:
hsd_imaging(
    file_name="data.ouput.sd.du",
    processing_set=processing_set, 
    leaves=["uid___A002_Xc1469f_X1c33_65"]
)

### Example: Linked Graphs

In [ ]:
def first_function():
    pass

def second_function():
    pass

def gather():
    pass

def connected_graph(
    file_name="connected.ouput.sd.du",
    processing_set=processing_set, 
    leaves=["uid___A002_Xc1469f_X1c33_65"]
):
    from xradio.measurement_set import open_processing_set
    
    ps = open_processing_set(
        ps_store=str(processing_set),
        scan_intents=["OBSERVE_TARGET#ON_SOURCE"]
    )

    job = toolviper.utils.sd.graph.Graph.from_dataset(ps)

    # If you want to filter sub-dataset
    if leaves:
        job.filter(leaves=leaves)

    # We are actually doing [field, spw, antenna] here but the processing set is already split up by spw.
    job.make_coordinates(coords=["antenna_name", "field_name"])
    job.build_node(ps_partition = ["spectral_window_name", "field_name"])
    
    # Start Dask processing - building job graph
    job.map(
        function=first_function, 
        parameters={
            "file_name": file_name
        },
        connect=None, 
        make_workflow=False
    )

    job.make_coordinates(coords=["antenna_name", "field_name"])
    job.build_node(ps_partition = ["spectral_window_name", "field_name", "antenna_name"])
    
    job.map(
        function=second_function, 
        parameters={
            "file_name": file_name
        },
        connect=first_function, 
        make_workflow=False
    )
    
    job.reduce(
        parameters={"delay": 'PM03_T701'}, 
        function=gather, 
        mode="single_node"
    )

    return job


In [ ]:
j = connected_graph()

In [ ]:
j.visualize()

In [ ]:
client.close()